In [0]:
# ==========================================================
# BRONZE LAYER (Raw ingestion)
#
#
# Only thing we add:
# - ingestion timestamp
# - source file name
# - batch id (to track which run this came from)
#
# Batch logic:
# - Batch 1 → full load (dimensions + some facts)
# - Batch 2-4 → only new fact data (incremental)
# ==========================================================

from pyspark.sql.functions import current_timestamp, lit, input_file_name


# --- S3 config  ---

BUCKET = "capstone-ecomm-team8"
S3 = f"s3a://{BUCKET}"


# Try getting batch number from Airflow
# If not present, default to 1 (useful when running manually)
try:
    batch_number = dbutils.widgets.get("batch_number")
except:
    dbutils.widgets.text("batch_number", "1")
    batch_number = "1"

print(f"Running Bronze ingestion for batch: {batch_number}")


def ingest_to_bronze(s3_path, table_name, mode="append"):
    """
    Reads CSV from S3 → adds basic audit columns → saves as Delta table.

    mode:
    - overwrite → first time load
    - append → incremental load
    """

    print(f"\nLoading → bronze.{table_name}")
    print(f"Source → {s3_path}")

    try:
        df = spark.read \
            .option("header", True) \
            .option("inferSchema", True) \
            .csv(s3_path)
    except Exception as e:
        print(f"Failed to read {s3_path}: {e}")
        return 0

    # Add basic tracking columns (very useful later for debugging)
    df = df.withColumn("ingestion_timestamp", current_timestamp()) \
           .withColumn("source_file", lit(s3_path.split("/")[-1])) \
           .withColumn("batch_id", lit(f"batch_{batch_number}"))

    row_count = df.count()

    # Save as Delta table
    df.write.format("delta").mode(mode).saveAsTable(f"bronze.{table_name}")

    print(f"Done → {row_count:,} rows written ({mode})")

    return row_count


total_rows = 0


# ==========================================================
# BATCH 1 → First run
#
# - Load all dimension tables (one-time load)
# - Load initial chunk of fact tables
#
# Everything is overwrite because this is the first load
# ==========================================================
if batch_number == "1":

    print("\n" + "="*60)
    print("Batch 1 → Full load (dims + initial facts)")
    print("="*60)

    # --- Dimensions (only loaded once) ---
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/product_category_name_translation.csv",
        "category_translation", "overwrite")

    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/geolocation_dataset.csv",
        "geolocation", "overwrite")

    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/sellers_dataset.csv",
        "sellers", "overwrite")

    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/customers_dataset.csv",
        "customers", "overwrite")

    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/products_dataset.csv",
        "products", "overwrite")

    # --- Facts (initial 25%) ---
    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/orders_dataset.csv",
        "orders", "overwrite")

    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/order_items_dataset.csv",
        "order_items", "overwrite")

    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/order_payments_dataset.csv",
        "order_payments", "overwrite")

    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_1/order_reviews_dataset.csv",
        "order_reviews", "overwrite")


# ==========================================================
# BATCH 2, 3, 4 → Incremental runs
#
# - Only new fact data comes in
# - We append to existing tables
# ==========================================================
elif batch_number in ["2", "3", "4"]:

    print("\n" + "="*60)
    print(f"Batch {batch_number} → Incremental load (facts only)")
    print("="*60)

    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_{batch_number}/orders_dataset.csv",
        "orders", "append")

    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_{batch_number}/order_items_dataset.csv",
        "order_items", "append")

    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_{batch_number}/order_payments_dataset.csv",
        "order_payments", "append")

    total_rows += ingest_to_bronze(
        f"{S3}/raw/batch_{batch_number}/order_reviews_dataset.csv",
        "order_reviews", "append")


# Final summary (helps a lot while debugging runs)
print("\n" + "="*60)
print(f"Bronze ingestion completed for batch {batch_number}")
print(f"Total rows ingested: {total_rows:,}")
print("="*60)